In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\fireprotdb_ddg_final_all_ver2.csv", low_memory=False)

In [2]:
df_mega = df[df["SOURCE_DATASET"]=="MegaScale"].copy()

In [3]:
print(df_mega)

       PROTEIN_ID WT  POS MUT  DDG_mean SOURCE_DATASET UNIPROTKB PDB_ID  \
0            1A0N  A   12   C -1.271679      MegaScale       NaN    NaN   
1            1A0N  A   12   D -2.513681      MegaScale       NaN    NaN   
2            1A0N  A   12   E -2.120722      MegaScale       NaN    NaN   
3            1A0N  A   12   F -2.238222      MegaScale       NaN    NaN   
4            1A0N  A   12   G -0.142810      MegaScale       NaN    NaN   
...           ... ..  ...  ..       ...            ...       ...    ...   
347020    v2_6IVS  Y    7   R  0.056541      MegaScale       NaN    NaN   
347021    v2_6IVS  Y    7   S -0.514305      MegaScale       NaN    NaN   
347022    v2_6IVS  Y    7   T -0.211927      MegaScale       NaN    NaN   
347023    v2_6IVS  Y    7   V -0.062451      MegaScale       NaN    NaN   
347024    v2_6IVS  Y    7   W  0.223641      MegaScale       NaN    NaN   

       MEGASCALE  
0           1A0N  
1           1A0N  
2           1A0N  
3           1A0N  
4   

In [4]:
import os
import pandas as pd

# 1. PDB 저장 경로 설정 (사용자 경로)
PDB_DIR = r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\AlphaFold_model_PDBs"

# 2. 파일 경로 생성 함수
def get_corrected_pdb_path(mega_id):
    # | 기호를 _로 치환 (예: EA|run2 -> EA_run2)
    clean_name = str(mega_id).replace('|', '_')
    path = os.path.join(PDB_DIR, f"{clean_name}.pdb")
    
    if os.path.exists(path):
        return path
    return None

# 3. 유니크한 MEGASCALE ID 기준으로 파일 존재 여부 확인 (효율성)
unique_mega = pd.DataFrame(df_mega['MEGASCALE'].unique(), columns=['MEGASCALE'])
unique_mega['FILE_PATH'] = unique_mega['MEGASCALE'].apply(get_corrected_pdb_path)

# 4. 원본 df_mega에 경로 정보 병합
df_mega = pd.merge(df_mega, unique_mega, on='MEGASCALE', how='left')

# 5. 기초 통계 확인
found_count = df_mega['FILE_PATH'].notna().sum()
print(f"전체 로우: {len(df_mega)} 개")
print(f"로컬에서 찾은 PDB 파일: {found_count} 개 ({(found_count/len(df_mega))*100:.2f}%)")

전체 로우: 342675 개
로컬에서 찾은 PDB 파일: 342675 개 (100.00%)


In [5]:
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import three_to_one, is_aa

def verify_mega_mapping(row):
    if pd.isna(row['FILE_PATH']):
        return "No_File", None
    
    target_pos = int(row['POS'])
    target_wt = row['WT']
    
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('protein', row['FILE_PATH'])
        model = structure[0]
        
        # 모든 체인을 돌며 해당 POS 확인
        for chain in model:
            if target_pos in chain:
                res = chain[target_pos]
                if is_aa(res):
                    res_name_1 = three_to_one(res.get_resname())
                    if res_name_1 == target_wt:
                        return "Match", chain.id
        
        return "Mismatch_or_Not_Found", None
    except Exception as e:
        return f"Error: {str(e)}", None

# 적용 (수량이 많으므로 진행 상황 확인을 위해 tqdm 추천)
# !pip install tqdm
from tqdm import tqdm
tqdm.pandas()

print("MegaScale WT-POS 매핑 검증 시작...")
# 유니크한 (MEGASCALE, POS, WT) 조합에 대해서만 먼저 돌리면 훨씬 빠릅니다.
unique_check = df_mega[['MEGASCALE', 'POS', 'WT', 'FILE_PATH']].drop_duplicates()
unique_check[['MAPPING_STATUS', 'MATCHED_CHAIN']] = unique_check.progress_apply(
    lambda x: pd.Series(verify_mega_mapping(x)), axis=1
)

# 결과를 다시 df_mega에 병합
df_mega = pd.merge(df_mega, unique_check, on=['MEGASCALE', 'POS', 'WT', 'FILE_PATH'], how='left')

MegaScale WT-POS 매핑 검증 시작...


100%|██████████| 19254/19254 [04:59<00:00, 64.29it/s] 


In [6]:
# 1. 'Match'인 것만 추출
df_mega_cleaned = df_mega[df_mega['MAPPING_STATUS'] == 'Match'].copy()

# 2. 결과 리포트
print("=== [ MegaScale 정제 결과 ] ===")
print(f"최종 남은 로우: {len(df_mega_cleaned):,} 개")
print(f"제거된 로우: {len(df_mega) - len(df_mega_cleaned):,} 개")

# 3. 실패 원인 통계
print("\n[ 실패 원인 분포 ]")
print(df_mega['MAPPING_STATUS'].value_counts())

=== [ MegaScale 정제 결과 ] ===
최종 남은 로우: 341,443 개
제거된 로우: 1,232 개

[ 실패 원인 분포 ]
MAPPING_STATUS
Match                    341443
Mismatch_or_Not_Found      1232
Name: count, dtype: int64


In [7]:
df_mega[df_mega['MAPPING_STATUS']!='Match']["MEGASCALE"].value_counts()

MEGASCALE
2HBB    98
2B88    62
1PGA    60
1UBQ    54
2K5H    38
2KYB    38
1H8K    37
2KZI    36
5FWB    36
1MJC    36
1PGX    36
2LXK    36
1A0N    35
2LYP    35
2MKX    35
5ECA    35
2KT8    34
2ROT    34
2K52    34
1BK2    34
1CSQ    33
1S1N    33
2KRS    32
2ZW1    31
2LSS    30
1OPS    28
1EM7    26
1SIF    25
2PTL    24
1GB4    21
2M7O    19
6NS8    19
1YEZ    19
1SF0    17
5XR0    11
1UCS     9
2LX2     5
5GU9     4
2MLB     3
Name: count, dtype: int64

In [8]:
df_mega[(df_mega["MEGASCALE"]=="2HBB")]

,PROTEIN_ID,WT,POS,MUT,DDG_mean,SOURCE_DATASET,UNIPROTKB,PDB_ID,MEGASCALE,FILE_PATH,MAPPING_STATUS,MATCHED_CHAIN
71127,2HBB,A,22,C,-0.153565,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71128,2HBB,A,22,D,0.425834,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71129,2HBB,A,22,F,-0.546964,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71130,2HBB,A,22,G,-0.438680,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71131,2HBB,A,22,H,-0.095823,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
...,...,...,...,...,...,...,...,...,...,...,...,...
71977,2HBB,Y,25,R,-1.459033,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71978,2HBB,Y,25,S,-1.754541,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71979,2HBB,Y,25,T,-1.554375,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71980,2HBB,Y,25,V,-1.105048,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A


In [10]:
# 1. 단백질별 매핑 성공률 계산
protein_stats = df_mega.groupby('MEGASCALE')['MAPPING_STATUS'].apply(
    lambda x: (x == 'Match').mean()
).reset_index(name='success_rate')

# 2. 신뢰할 수 없는 단백질 리스트 (예: 성공률 95% 미만)
THRESHOLD = 0.95
bad_proteins = protein_stats[protein_stats['success_rate'] < THRESHOLD]['MEGASCALE'].unique()

print(f"❌ 신뢰도 미달로 퇴출될 단백질 수: {len(bad_proteins)} 개")
print(f"퇴출 목록 샘플: {bad_proteins[:10]}")

# 3. 최종 필터링
# - 퇴출 단백질에 속하지 않으면서
# - 개별 Row 상태도 'Match'인 것만 유지
df_mega_ultra_clean = df_mega[
    (~df_mega['MEGASCALE'].isin(bad_proteins)) & 
    (df_mega['MAPPING_STATUS'] == 'Match')
].copy()

print(f"✅ 최종 생존 Row: {len(df_mega_ultra_clean):,} 개")

❌ 신뢰도 미달로 퇴출될 단백질 수: 3 개
퇴출 목록 샘플: ['1PGA' '2B88' '2HBB']
✅ 최종 생존 Row: 338,995 개


In [22]:
# 1. 단백질별 매핑 성공률 계산
protein_stats = df_mega.groupby('MEGASCALE')['MAPPING_STATUS'].apply(
    lambda x: (x == 'Match').mean()
).reset_index(name='success_rate')

# 2. 신뢰할 수 없는 단백질 리스트 (예: 성공률 95% 미만)
THRESHOLD = 1.0
bad_proteins = protein_stats[protein_stats['success_rate'] < THRESHOLD]['MEGASCALE'].unique()

print(f"❌ 신뢰도 미달로 퇴출될 단백질 수: {len(bad_proteins)} 개")
print(f"퇴출 목록 샘플: {bad_proteins[:10]}")

# 3. 최종 필터링
# - 퇴출 단백질에 속하지 않으면서
# - 개별 Row 상태도 'Match'인 것만 유지
df_mega_ultra_clean = df_mega[
    (~df_mega['MEGASCALE'].isin(bad_proteins)) & 
    (df_mega['MAPPING_STATUS'] == 'Match')
].copy()

print(f"✅ 최종 생존 Row: {len(df_mega_ultra_clean):,} 개")

❌ 신뢰도 미달로 퇴출될 단백질 수: 39 개
퇴출 목록 샘플: ['1A0N' '1BK2' '1CSQ' '1EM7' '1GB4' '1H8K' '1MJC' '1OPS' '1PGA' '1PGX']
✅ 최종 생존 Row: 301,709 개


In [23]:
df_mega_ultra_clean

,PROTEIN_ID,WT,POS,MUT,DDG_mean,SOURCE_DATASET,UNIPROTKB,PDB_ID,MEGASCALE,FILE_PATH,MAPPING_STATUS,MATCHED_CHAIN
1077,1A32,A,45,C,0.382630,MegaScale,NaN,NaN,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
1078,1A32,A,45,D,-0.012520,MegaScale,NaN,NaN,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
1079,1A32,A,45,E,0.259534,MegaScale,NaN,NaN,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
1080,1A32,A,45,F,-0.290754,MegaScale,NaN,NaN,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
1081,1A32,A,45,G,-0.221574,MegaScale,NaN,NaN,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
...,...,...,...,...,...,...,...,...,...,...,...,...
342670,v2_6IVS,Y,7,R,0.056541,MegaScale,NaN,NaN,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
342671,v2_6IVS,Y,7,S,-0.514305,MegaScale,NaN,NaN,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
342672,v2_6IVS,Y,7,T,-0.211927,MegaScale,NaN,NaN,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
342673,v2_6IVS,Y,7,V,-0.062451,MegaScale,NaN,NaN,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A


In [24]:
df_mega_ultra_clean = df_mega_ultra_clean.drop(columns=["PROTEIN_ID","PDB_ID","SOURCE_DATASET","UNIPROTKB","MAPPING_STATUS"])
df_mega_ultra_clean

,WT,POS,MUT,DDG_mean,MEGASCALE,FILE_PATH,MATCHED_CHAIN
1077,A,45,C,0.382630,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
1078,A,45,D,-0.012520,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
1079,A,45,E,0.259534,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
1080,A,45,F,-0.290754,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
1081,A,45,G,-0.221574,1A32,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
...,...,...,...,...,...,...,...
342670,Y,7,R,0.056541,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
342671,Y,7,S,-0.514305,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
342672,Y,7,T,-0.211927,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A
342673,Y,7,V,-0.062451,v2_6IVS,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,A


In [25]:
import os

# FILE_PATH 컬럼에서 파일명만 추출하여 덮어쓰기
df_mega_ultra_clean['FILE_PATH'] = df_mega_ultra_clean['FILE_PATH'].apply(lambda x: os.path.basename(x))

df_mega_ultra_clean

,WT,POS,MUT,DDG_mean,MEGASCALE,FILE_PATH,MATCHED_CHAIN
1077,A,45,C,0.382630,1A32,1A32.pdb,A
1078,A,45,D,-0.012520,1A32,1A32.pdb,A
1079,A,45,E,0.259534,1A32,1A32.pdb,A
1080,A,45,F,-0.290754,1A32,1A32.pdb,A
1081,A,45,G,-0.221574,1A32,1A32.pdb,A
...,...,...,...,...,...,...,...
342670,Y,7,R,0.056541,v2_6IVS,v2_6IVS.pdb,A
342671,Y,7,S,-0.514305,v2_6IVS,v2_6IVS.pdb,A
342672,Y,7,T,-0.211927,v2_6IVS,v2_6IVS.pdb,A
342673,Y,7,V,-0.062451,v2_6IVS,v2_6IVS.pdb,A


In [26]:
df_mega_ultra_clean = df_mega_ultra_clean.rename(columns={
    "MUT": "Mut",
    "POS": "MutPos",
    "DDG_mean": "DDG",
    "MEGASCALE": "PDB_ID",
    "MATCHED_CHAIN": "Chain"
})

In [31]:
df_mega_ultra_clean

,WT,MutPos,Mut,DDG,PDB_ID,FILE_PATH,Chain,SeqPos
1077,A,45,C,0.382630,1A32,1A32.pdb,A,NaN
1078,A,45,D,-0.012520,1A32,1A32.pdb,A,NaN
1079,A,45,E,0.259534,1A32,1A32.pdb,A,NaN
1080,A,45,F,-0.290754,1A32,1A32.pdb,A,NaN
1081,A,45,G,-0.221574,1A32,1A32.pdb,A,NaN
...,...,...,...,...,...,...,...,...
342670,Y,7,R,0.056541,v2_6IVS,v2_6IVS.pdb,A,NaN
342671,Y,7,S,-0.514305,v2_6IVS,v2_6IVS.pdb,A,NaN
342672,Y,7,T,-0.211927,v2_6IVS,v2_6IVS.pdb,A,NaN
342673,Y,7,V,-0.062451,v2_6IVS,v2_6IVS.pdb,A,NaN


In [ ]:
df_mega_ultra_clean[df_mega_ultra_clean["PDB_ID"]=="XX|run10_1081_0004_A"] 

,WT,MutPos,Mut,DDG,PDB_ID,FILE_PATH,Chain,SeqPos
1077,A,45,C,0.382630,1A32,1A32.pdb,A,NaN
1078,A,45,D,-0.012520,1A32,1A32.pdb,A,NaN
1079,A,45,E,0.259534,1A32,1A32.pdb,A,NaN
1080,A,45,F,-0.290754,1A32,1A32.pdb,A,NaN
1081,A,45,G,-0.221574,1A32,1A32.pdb,A,NaN
...,...,...,...,...,...,...,...,...
342670,Y,7,R,0.056541,v2_6IVS,v2_6IVS.pdb,A,NaN
342671,Y,7,S,-0.514305,v2_6IVS,v2_6IVS.pdb,A,NaN
342672,Y,7,T,-0.211927,v2_6IVS,v2_6IVS.pdb,A,NaN
342673,Y,7,V,-0.062451,v2_6IVS,v2_6IVS.pdb,A,NaN


In [35]:
df_mega_ultra_clean = df_mega_ultra_clean.reset_index(drop=True)

In [36]:
import pandas as pd
import os
from Bio.PDB import PDBParser, PPBuilder

def generate_s669_fasta_with_pos(df, pdb_dir, output_file):
    parser = PDBParser(QUIET=True)
    ppb = PPBuilder()
    
    # 3문자 -> 1문자 변환용 (PPBuilder 내부에서 쓰이지만 매핑 확인용으로 선언)
    d3to1 = {'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F', 'GLY': 'G', 'HIS': 'H', 
             'ILE': 'I', 'LYS': 'K', 'LEU': 'L', 'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 
             'ARG': 'R', 'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'}

    # 결과를 저장할 리스트 (나중에 df에 합침)
    seq_pos_list = [None] * len(df)
    
    # 유니크한 PDB-Chain 조합별로 처리
    unique_targets = df[['PDB_ID', 'Chain','FILE_PATH']].drop_duplicates()
    
    print(f"--- FASTA 생성 및 포지션 매핑 시작 ---")
    
    with open(output_file, 'w') as f_out:
        for _, target in unique_targets.iterrows():
            pdb_id, chain_id, file_path = target['PDB_ID'], target['Chain'], target["FILE_PATH"]
            pdb_path = os.path.join(pdb_dir, f"{file_path}")
            
            if not os.path.exists(pdb_path):
                continue
                
            try:
                structure = parser.get_structure(pdb_id, pdb_path)
                model = structure[0]
                if chain_id not in model: continue
                
                # 1. PPBuilder로 물리적으로 연결된 서열 조각(Polypeptides) 추출
                polypeptides = ppb.build_peptides(model[chain_id])
                
                full_seq = ""
                # FASTA 내 각 글자가 실제 PDB에서 몇 번이었는지 기록할 리스트
                pdb_res_nums = [] 

                for pp in polypeptides:
                    full_seq += str(pp.get_sequence())
                    # 각 잔기의 PDB 번호(id[1])를 순서대로 추출
                    for residue in pp:
                        pdb_res_nums.append(residue.get_id()[1])
                
                if not full_seq: continue

                # 2. FASTA 파일 쓰기
                f_out.write(f">{pdb_id}_{chain_id}\n{full_seq}\n")

                # 3. 해당 PDB_ID/Chain을 쓰는 모든 행(Mutation)에 대해 SeqPos 계산
                row_indices = df[(df['PDB_ID'] == pdb_id) & (df['Chain'] == chain_id)].index
                
                for idx in row_indices:
                    target_pdb_pos = int(df.at[idx, 'MutPos'])
                    
                    # PDB 번호 리스트에서 해당 번호가 몇 번째(index)에 있는지 찾기
                    try:
                        # 0-based index에 1을 더해 1-based position 생성
                        seq_index = pdb_res_nums.index(target_pdb_pos)
                        seq_pos_list[idx] = seq_index + 1 
                    except ValueError:
                        # PDB 구조에 좌표가 없는 번호인 경우
                        seq_pos_list[idx] = None
                        
            except Exception as e:
                print(f"Error in {pdb_id}_{chain_id}: {e}")

    # 4. 데이터프레임 업데이트
    df['SeqPos'] = seq_pos_list
    print(f"--- 완료! FASTA가 저장되었고 'SeqPos' 컬럼이 추가되었습니다. ---")
    return df

# --- 실행 부분 ---
pdb_dir_path = r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\AlphaFold_model_PDBs"
output_fasta = "Mega.fasta"

# 함수 실행 및 결과 업데이트
df_mega_ultra_clean = generate_s669_fasta_with_pos(df_mega_ultra_clean, pdb_dir_path, output_fasta)

# 결과 확인
print(df_mega_ultra_clean[['PDB_ID', 'Chain', 'MutPos', 'SeqPos', 'WT']].head())

--- FASTA 생성 및 포지션 매핑 시작 ---
--- 완료! FASTA가 저장되었고 'SeqPos' 컬럼이 추가되었습니다. ---
  PDB_ID Chain  MutPos  SeqPos WT
0   1A32     A      45      45  A
1   1A32     A      45      45  A
2   1A32     A      45      45  A
3   1A32     A      45      45  A
4   1A32     A      45      45  A


In [37]:
df_mega_ultra_clean

,WT,MutPos,Mut,DDG,PDB_ID,FILE_PATH,Chain,SeqPos
0,A,45,C,0.382630,1A32,1A32.pdb,A,45
1,A,45,D,-0.012520,1A32,1A32.pdb,A,45
2,A,45,E,0.259534,1A32,1A32.pdb,A,45
3,A,45,F,-0.290754,1A32,1A32.pdb,A,45
4,A,45,G,-0.221574,1A32,1A32.pdb,A,45
...,...,...,...,...,...,...,...,...
301704,Y,7,R,0.056541,v2_6IVS,v2_6IVS.pdb,A,7
301705,Y,7,S,-0.514305,v2_6IVS,v2_6IVS.pdb,A,7
301706,Y,7,T,-0.211927,v2_6IVS,v2_6IVS.pdb,A,7
301707,Y,7,V,-0.062451,v2_6IVS,v2_6IVS.pdb,A,7


In [39]:
df_mega_ultra_clean["Key"] = df_mega_ultra_clean["PDB_ID"] + "_" + df_mega_ultra_clean["Chain"]

In [40]:
df_mega_ultra_clean.to_csv("Mega.tsv", sep='\t', index=False)

In [45]:
import pandas as pd

def analyze_pos_differences(df):
    # 1. 오프셋 계산 (PDB번호 - FASTA순서)
    # SeqPos가 NaN인 경우(구조에 좌표가 없는 경우)는 제외하고 계산합니다.
    df_valid = df.dropna(subset=['SeqPos']).copy()
    df_valid['Offset'] = df_valid['MutPos'].astype(int) - df_valid['SeqPos'].astype(int)

    # 2. Key 단위로 그룹화하여 통계 산출
    key_stats = df_valid.groupby('Key')['Offset'].agg(['min', 'max', 'mean', 'nunique', 'std']).reset_index()
    
    # 3. 결과 분석 및 카운팅
    # 오프셋이 일정한 그룹 (std == 0 또는 nunique == 1)
    consistent_keys = key_stats[key_stats['nunique'] == 1]
    
    # 오프셋이 상이한 그룹 (중간에 Gap이 있거나 구조가 끊긴 경우)
    inconsistent_keys = key_stats[key_stats['nunique'] > 1]
    
    # [추가] 오프셋이 0이 아닌 케이스 계산
    # 1) 일정한데 0이 아닌 경우 (단순 오프셋 존재)
    nonzero_consistent = consistent_keys[consistent_keys['min'] != 0]
    # 2) 일정하지 않은 경우 (Gap 존재 등, 당연히 0이 아님)
    nonzero_total_count = len(nonzero_consistent) + len(inconsistent_keys)
    # 3) 완벽하게 1:1 매칭되는 경우 (Offset이 0인 경우)
    perfect_match_count = len(consistent_keys[consistent_keys['min'] == 0])

    print(f" 전체 요약 ")
    print(f"- 분석 대상 Key 개수: {len(key_stats)}")
    print(f"- 오프셋이 0인 Key (완벽 일치): {perfect_match_count}")
    print(f"- 오프셋이 0이 아닌 Key (오프셋/Gap 존재): {nonzero_total_count}") # 요청하신 부분
    print(f"  └ 오프셋이 일정하나 0은 아님: {len(nonzero_consistent)}")
    print(f"  └ 오프셋이 불규칙함 (Gap 의심): {len(inconsistent_keys)}")
    print("-" * 30)

    if not inconsistent_keys.empty:
        print("\n[ 오프셋이 상이한 Key 리스트 (주의 필요) ]")
        print(inconsistent_keys.sort_values('std', ascending=False).head(10))
    
    if not nonzero_consistent.empty:
        print("\n[ 오프셋이 일정하지만 0은 아닌 Key 샘플 ]")
        print(nonzero_consistent[['Key', 'min']].rename(columns={'min': 'Constant_Offset'}).head(10))

    return key_stats, inconsistent_keys

# 실행 예시
df_notMega = pd.read_csv(r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\Mega.tsv", sep="\t")
key_summary, gap_keys = analyze_pos_differences(df_notMega)

 전체 요약 
- 분석 대상 Key 개수: 332
- 오프셋이 0인 Key (완벽 일치): 332
- 오프셋이 0이 아닌 Key (오프셋/Gap 존재): 0
  └ 오프셋이 일정하나 0은 아님: 0
  └ 오프셋이 불규칙함 (Gap 의심): 0
------------------------------
